### Models

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
import math
import matplotlib
import matplotlib.pyplot as plt
import wandb

def positionalencoding2d(d_model, height, width):
    """
    :param d_model: dimension of the model
    :param height: height of the positions
    :param width: width of the positions
    :return: d_model*height*width position matrix
    """
    if d_model % 4 != 0:
        raise ValueError("Cannot use sin/cos positional encoding with "
                         "odd dimension (got dim={:d})".format(d_model))
    pe = torch.zeros(d_model, height, width)
    # Each dimension use half of d_model
    d_model = int(d_model / 2)
    div_term = torch.exp(torch.arange(0., d_model, 2) *
                         -(math.log(10000.0) / d_model))
    pos_w = torch.arange(0., width).unsqueeze(1)
    pos_h = torch.arange(0., height).unsqueeze(1)
    pe[0:d_model:2, :, :] = torch.sin(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
    pe[1:d_model:2, :, :] = torch.cos(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
    pe[d_model::2, :, :] = torch.sin(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
    pe[d_model + 1::2, :, :] = torch.cos(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)

    return pe

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer

class OutAttention(torch.nn.Module):
    def __init__(self, num_heads, embedding_dim, out_dim=64, 
                 tau=1.0, drop_prob=0.0):
        super(OutAttention, self).__init__()

        self.num_heads = num_heads
        self.out_dim = out_dim
        self.embedding_dim = embedding_dim

        self.d_k = out_dim // num_heads
        self.d_v = out_dim // num_heads

        self.Wv = torch.nn.Linear(out_dim, out_dim)

        self.Q = torch.rand((1, out_dim), device='cuda')

        self.layernorm = torch.nn.LayerNorm(out_dim)

        self.linear_out = nn.Sequential(
            nn.Linear(out_dim, 2*out_dim),
            nn.Dropout(drop_prob),
            nn.ReLU(),
            nn.Linear(2*out_dim, out_dim)
        )

    def forward(self, embeddings, generated_weights):
        batch = embeddings.size(0)
        seq_len = embeddings.size(1)
        embeddings = self.layernorm(embeddings)
        
        Wq,Wk = generated_weights
        Q = torch.matmul(self.Q.repeat(batch, 1, 1),Wq)  # (batch, seq_len, embedding_dim)
        assert Q.shape == (batch, 1, self.out_dim), f'{Q.shape}, {(batch, 1, self.out_dim)}'
        K = torch.matmul(embeddings,Wk)
        V = self.Wv(embeddings)

        # Converting to multiheaded attention
        Q = Q.view(batch, 1, self.num_heads, self.d_k).transpose(1,2)  # (batch, num_heads, seq_len, d_k)
        assert Q.shape == (batch, self.num_heads, 1, self.d_k), f'{Q.shape}, {(batch, self.num_heads, seq_len, self.d_k)}'
        K = K.view(batch, seq_len, self.num_heads, self.d_k).transpose(1,2)  
        V = V.view(batch, seq_len, self.num_heads, self.d_v).transpose(1,2)  

        # Compute attention scores
        attention_scores = Q @ K.transpose(2,3)  # (num_heads, seq_len, seq_len)
        assert attention_scores.shape ==(batch, self.num_heads, 1, seq_len)

        # Scale the attention scores
        attention_scores = attention_scores / math.sqrt(self.d_k)

        # Apply softmax to get attention weights
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = torch.nan_to_num(attention_weights, nan=0.0)
        assert attention_weights.shape == (batch, self.num_heads, 1, seq_len)

        self.attention_weights = attention_weights

        # Compute the output (weighted sum of values)
        output = attention_weights @ V
        assert output.shape == (batch, self.num_heads, 1, self.d_v), \
            f'{output.shape}, {(batch, self.num_heads, 1, self.d_v)}'

        output = output.transpose(1,2)
        output = output.flatten(start_dim=-2).squeeze(1)
        assert output.shape == (batch, self.out_dim)

        output = self.linear_out(output)
        return output

class ConditionedMultiHeadAttention(torch.nn.Module):
    def __init__(self, weight_gen_input_dim=2,dim=32,drop_prob=0.0):
        super(ConditionedMultiHeadAttention, self).__init__()
        self.dim = dim
        self.d_k = dim

        # One shared Wv
        self.Wv = nn.Linear(dim, dim)

        # Output FFNs
        self.linear1 = nn.Sequential(
            nn.Linear(dim, 2*dim),
            nn.Dropout(drop_prob),
            nn.ReLU(),
            nn.Linear(2*dim, dim)
        )
        self.linear2 = nn.Sequential(
            nn.Linear(dim, 2*dim),
            nn.Dropout(drop_prob),
            nn.ReLU(),
            nn.Linear(2*dim, dim)
        )

        self.layernorm1 = torch.nn.LayerNorm(dim)
        self.layernorm2 = torch.nn.LayerNorm(dim)

        self.dropout1 = torch.nn.Dropout(p=drop_prob)
        self.dropout2 = torch.nn.Dropout(p=drop_prob)
    
    def forward(self, embeddings, generated_weights):
        batch = embeddings.size(0)
        seq_len = embeddings.size(1)

        # Save for skip connection
        x0 = embeddings

        embeddings = self.layernorm1(embeddings)

        # Use the generated Wq, Wk matrices to compute Q and K
        Wq, Wk = generated_weights
        Q = torch.matmul(embeddings,Wq)  # (batch, seq_len, embedding_dim)
        assert Q.shape == (batch, seq_len, self.dim)
        K = torch.matmul(embeddings,Wk)
        V = self.Wv(embeddings)

        # Compute attention scores
        attention_scores = Q @ K.transpose(1,2) / np.sqrt(self.d_k)  # (num_heads, seq_len, seq_len)
        assert attention_scores.shape ==(batch, seq_len, seq_len)
        
        attention_weights = torch.softmax(attention_scores, dim=-1)
        assert attention_weights.shape == (batch, seq_len, seq_len)
        self.attention_weights = attention_weights

        # Compute the output (weighted sum of values)
        output = attention_weights @ V
        assert output.shape == (batch, seq_len, self.dim)
        
        att_out = x0 + output
        mlp_out = self.linear1(self.layernorm2(att_out))
        att_out = att_out + mlp_out

        return att_out

class WeightGenerator(nn.Module):
    # Generates the Wq and Wk matrices for the attention network.
    def __init__(self, num_layers=1,input_dim=2,dim=32):
        super(WeightGenerator, self).__init__()
        self.num_layers = num_layers
        self.dim = dim
        self.input_dim=input_dim

        self.net = nn.Sequential(
                        nn.Linear(self.input_dim, 4*self.dim),
                        nn.ReLU(),
                        nn.Linear(4*self.dim, 8*self.dim),
                        nn.ReLU(),
                        nn.Linear(8*self.dim, (2*(self.num_layers+1) * self.dim*self.dim))
                    )
    def forward(self, z):
        b = z.shape[0]
        return self.net(z).reshape(b,2*(self.num_layers+1),self.dim,self.dim).permute(1,0,2,3)


class TransformerAgent(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.dim = args.dim
        self.temp = args.temp
        self.weight_gen_input_dim = args.weight_gen_input_dim

        # Given an input, generates a Wq and Wk matrix for each attention layer, 
        # and a Wq,Wk matrix for the attention pooling layer.
        self.weight_generator = WeightGenerator(args.num_layers,args.weight_gen_input_dim,args.dim)

        # Token Extractor
        self.encoder = nn.Embedding(num_embeddings=args.num_embeddings, 
                                    embedding_dim=args.dim)

        self.w = args.width
        # Calculate the positional encodings
        sin_pos = positionalencoding2d(d_model=args.dim, height=self.w, width=self.w).to('cuda')
        sin_pos = torch.permute(sin_pos, (1,2,0)).flatten(start_dim=0, end_dim=1).unsqueeze(0)
        assert sin_pos.shape == (1, self.w*self.w, args.dim)
        self.pos = sin_pos

        # Define attention layers
        self.num_layers = args.num_layers
        self.MHA_layers = torch.nn.ModuleList([ConditionedMultiHeadAttention(
                                    weight_gen_input_dim=args.weight_gen_input_dim,dim=self.dim) 
                                    for _ in range(self.num_layers)])
        
        # Define aggregation function
        if args.aggregation == 'attention':
            self.agg_out = OutAttention(num_heads=1, 
                                        embedding_dim=self.dim, 
                                        out_dim=args.dim)
        elif args.aggregation == 'mean':
            self.agg_out = lambda x: (x.mean(dim=-2), torch.ones(x.shape[1], device='cuda'), torch.ones(x.shape[1], device='cuda'), torch.ones(x.shape[1], device='cuda'))
        
        # Get max eta values for normalization
        eta = torch.ones((self.w*self.w,self.w*self.w), device='cuda')
        for layer in self.MHA_layers[1:]:
            eta = torch.ones((self.w*self.w,self.w*self.w), device='cuda') @ eta
        eta = torch.ones(self.w*self.w, device='cuda') @ eta
        self.max_eta = eta
    
        # Final output network
        self.out_net = nn.Sequential(nn.Linear(args.dim, 256), nn.ReLU(), layer_init(nn.Linear(256, 2), std=0.01))

    def hidden(self, x, z):
        batch_size, _ = x.shape
        dim = self.dim
        w = self.w
        pos = self.pos.repeat((batch_size,1,1))

        # Extract feature tokens via CNN
        f = self.encoder(x)
        assert f.shape == (batch_size, w*w, dim), f'{f.shape}, {(batch_size, w*w, dim)}'
        
        # Add positional encodings
        f = f + pos

        # Attention layers
        w = self.weight_generator(z)
        assert w.shape[0] == 2*(self.num_layers+1), w.shape
        out = f
        for i, layer in enumerate(self.MHA_layers):
            out = layer(out, w[2*i : 2*i+2])
            self.attention_weights = layer.attention_weights
        # Attention Aggregation
        out = self.agg_out(out,w[-2:])
        self.out_weights = self.agg_out.attention_weights
        return out

    def forward(self, x, z=None):
        h = self.hidden(x, z)
        logits = self.out_net(h)
        return logits
    
    def get_eta(self):
        o = self.MHA_layers[0].attention_weights
        for layer in self.MHA_layers[1:]:
            o = layer.attention_weights @ o
        out_w = self.agg_out.attention_weights.squeeze(1)
        o = out_w @ o
        return o/self.max_eta[0]


### Train

In [ ]:
import numpy as np
import torch
from torch import nn
import dill
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from sparse_generalization.utils.dataloading import get_shapes_datasets

class Args:
    batch_size:int = 256
    epochs:int = 500
    learning_rate:float = 5e-5
    max_grad_norm:float = 100.0

    val_fraction:float = 0.1
    num_val:int = 100
    num_train:int = 2000

    diversity_coef: float = 100.0
    sparsity_coef: float = 0.0

    weight_gen_input_dim:int = 2

    width:int = 4
    num_embeddings:int = 16

    num_layers:int = 3
    dim:int = 32
    temp:float = 1.0
    aggregation:str = 'attention'

def train(args):
    args.num_embeddings = args.width * args.width

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    # Load training data and both version of testing data.
    with open(f"shape_data/shapes_train_5000_size{args.width}.pkl", "rb") as f:
        data = dill.load(f)
        X = data['X_train'].flatten(start_dim=1)
        y = data['Y_train'].squeeze(-1)
    TEST_Xs = []; TEST_Ys = []
    for kind in ['test_a','test_b']:
        with open(f"shape_data/shapes_{kind}_5000_size{args.width}.pkl", "rb") as f:
            data = dill.load(f)
            TEST_Xs.append(data[f'X_{kind}'])
            TEST_Ys.append(data[f'Y_{kind}'].squeeze(-1))
    # Convert labels to one-hot
    unique_labels, inverse_indices = torch.unique(y, return_inverse=True)
    num_classes = len(unique_labels)
    y = torch.eye(num_classes)[inverse_indices]

    TEST_X = [d.flatten(start_dim=1) for d in TEST_Xs]
    TEST_y = []
    for d in TEST_Ys:
        unique_labels, inverse_indices = torch.unique(d, return_inverse=True)
        num_classes = len(unique_labels)
        one_hot = torch.eye(num_classes)[inverse_indices]
        TEST_y.append(one_hot)

    print("X shape:", X.shape)
    print("y shape:", y.shape)


    # Define datasets
    dataset = TensorDataset(X, y)
    test_datasets = [TensorDataset(tx,ty) for (tx,ty) in zip(TEST_X,TEST_y)]

    n_val = args.num_val
    n_train = args.num_train

    # Gather train and validation data from the training data
    train_dataset, val_dataset, _ = random_split(
        dataset,
        [n_train, n_val, len(dataset)-(n_train+n_val)],
        generator=torch.Generator(),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
    )
    test_loaders = [DataLoader(
        ds,
        batch_size=args.batch_size,
        shuffle=False,
    ) for ds in test_datasets]


    # Model
    model=TransformerAgent(args).to(DEVICE)

    # Loss & Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=args.learning_rate,
    )


    # Histories and plotting
    accuracy_history = []
    val_accuracy_history = []
    val_x = []
    fig, ax = plt.subplots()

    # Training
    for epoch in range(args.epochs):
        # Linear LR annealing
        for param_group in optimizer.param_groups:
            param_group['lr'] = args.learning_rate * (1-(epoch/args.epochs))
        model.train()

        train_loss = 0.0
        acc = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.float().to(DEVICE)
            optimizer.zero_grad()

            # First Category/Condition
            z = torch.ones(X_batch.shape[0]).long()
            z = nn.functional.one_hot(z, num_classes=args.weight_gen_input_dim).float().to('cuda')
            predictions = model(X_batch,z).squeeze(-1)
            # The eta (influence from each input)
            w1 = model.get_eta()
            loss1 = criterion(predictions, y_batch)
            acc+=(predictions.argmax(dim=1) == y_batch.argmax(dim=1)).float().sum().item()

            # Second Category/Condition
            z = torch.zeros(X_batch.shape[0]).long()
            z = nn.functional.one_hot(z, num_classes=args.weight_gen_input_dim).float().to('cuda')
            predictions = model(X_batch,z).squeeze(-1)
            # The eta (influence from each input)
            w2 = model.get_eta()
            loss2 = criterion(predictions, y_batch)
            acc+=(predictions.argmax(dim=1) == y_batch.argmax(dim=1)).float().sum().item()

            # Diversity loss encourages the model to use different inputs for different solutions.
            diversity_loss = -torch.abs(w1-w2).sum(dim=-1).mean()

            # Sparse loss has not been necessary so far. This is a basic one. 
            sparse_loss = torch.linalg.vector_norm(w1.sum(dim=-1).squeeze(-1), ord=0.5,dim=-1).mean() + \
                torch.linalg.vector_norm(w2.sum(dim=-1).squeeze(-1), ord=0.5, dim=-1).mean()
            
            loss = loss1 + loss2 + (args.diversity_coef*diversity_loss) + (args.sparsity_coef*sparse_loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)
            optimizer.step()

        acc /= 2*len(train_loader.dataset)
        accuracy_history.append(acc)

        # Every once in a while, log the accurac on the validation data.
        val_acc=0.0
        with torch.no_grad():
            if epoch%100 == 0:
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(DEVICE)
                    y_batch = y_batch.float().to(DEVICE)
                    z = torch.randint(0, args.weight_gen_input_dim, (len(X_batch),))
                    z = nn.functional.one_hot(z, num_classes=args.weight_gen_input_dim).float().to('cuda')
                    predictions = model(X_batch,z).squeeze(-1)
                    val_acc += (predictions.argmax(dim=1) == y_batch.argmax(dim=1)).float().sum().item()
                val_acc /= len(val_loader.dataset)
                val_accuracy_history.append(val_acc)
                val_x.append(epoch)
        # Update the plot once in a while
        if epoch % 200 == 0:
            ax.cla() 
            ax.plot(accuracy_history)
            ax.plot(val_x, val_accuracy_history)
            clear_output(wait=True)
            display(fig)

        train_loss /= len(train_loader.dataset)
        

    # Validation and Test Data Final Scores
    model.eval()

    val_acc=0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.float().to(DEVICE)
            z = torch.randint(0, args.weight_gen_input_dim, (len(X_batch),))
            z = nn.functional.one_hot(z, num_classes=args.weight_gen_input_dim).float().to('cuda')
            predictions = model(X_batch,z).squeeze(-1)
            val_acc += (predictions.argmax(dim=1) == y_batch.argmax(dim=1)).float().sum().item()
    val_acc /= len(val_loader.dataset)

    # Test the model under both conditions on both sets of test data,
    # (each set of test data has a different solution)
    test_accs = []
    with torch.no_grad():
        for c in [0,1]:
            for i,test_loader in enumerate(test_loaders):
                acc = 0.0
                for X_batch, y_batch in test_loader:
                    X_batch = X_batch.to(DEVICE)
                    y_batch = y_batch.float().to(DEVICE)
                    z = (torch.ones((len(X_batch),))*c).long()
                    z = nn.functional.one_hot(z, num_classes=args.weight_gen_input_dim).float().to('cuda')
                    predictions = model(X_batch,z).squeeze(-1)
                    acc+=(predictions.argmax(dim=1) == y_batch.argmax(dim=1)).float().sum().item()
                test_accs.append(acc)
    test_accs = [t/len(test_loader.dataset) for t in test_accs]

    # Optionally, save the model 
    # torch.save(model.state_dict(), "model.pt")
    # print("Saved model.pt")

    return val_acc, test_accs